# 원판 탐지 보고서와 XAI 검수

기존 학습의 결과와 가중치를 사용합니다. 데이터 분할과 표본 편차를 확인한 뒤, 모델이 원판을 찾았는지와 중심을 얼마나 정확히 잡았는지 살펴봅니다. 생성한 그림은 선택한 학습 실행의 `report/` 폴더에 저장합니다.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image as NotebookImage
from PIL import Image
import numpy as np
from ultralytics import YOLO

from wlcv.config import DATASET, RUNS, SPLIT
from wlcv.data.plate_labels import read_boxes
from wlcv.evaluation.fiftyone_review import evaluate_plates, load_validation_frames
from wlcv.evaluation.occlusion import plot_occlusion_comparison
from wlcv.evaluation.report import (
    audit_dataset, collect_detections, compare_thresholds, plot_data_coverage,
    plot_frame_errors, plot_training_report,
)

run_dir = RUNS / "plate_baseline"
report_dir = run_dir / "report"
thresholds = [0.25, 0.10]
minimum_confidence = 0.001

# 후속 모델이나 앙상블을 비교할 때 모든 멤버의 학습 실행 폴더를 추가합니다.
training_runs = [run_dir]

def save_and_show(figure, filename):
    report_dir.mkdir(exist_ok=True)
    figure.savefig(report_dir / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)

## 1. 사람별 분할과 데이터 분포

사람별 라벨 프레임 수, 원판 박스 면적, 프레임 평균 밝기를 봅니다. 같은 영상의 여러 프레임은 서로 비슷할 수 있으므로 프레임 수와 사람 수를 같이 확인합니다.

선택한 각 학습 실행의 `train_people.csv`에서 현재 검증 사람과 겹치는지 확인합니다. 프레임 표의 해시는 전체 RGB 픽셀이 일치하는 학습·검증 프레임을 찾습니다. 비슷한 장면이나 편집된 복제 영상은 별도로 살펴봐야 합니다.

In [ ]:
frames, run_audit, duplicates = audit_dataset(DATASET, SPLIT, training_runs)
display(frames.groupby("split").agg(frames=("frame", "size"), people=("person_id", "nunique")))
display(run_audit)
print("학습·검증 양쪽에 동일한 픽셀이 있는 프레임 행 수", len(duplicates))
if not duplicates.empty:
    display(duplicates)

save_and_show(plot_data_coverage(frames), "data_coverage.png")

## 2. 학습 곡선과 탐지 성능

손실은 박스 위치·분류 등 항목별로 학습과 검증을 비교합니다. 검증 mAP는 박스 매칭 품질을 요약합니다. 학습률은 실제 에폭별 로그입니다.

PR 곡선은 재현율을 높일 때 정밀도가 어떻게 달라지는지 보여줍니다. 혼동 행렬은 Ultralytics가 실행할 때 저장한 탐지 결과입니다. 아래 FiftyOne 표와는 평가 설정이 다를 수 있습니다. 이 검증셋은 원판이 있는 프레임으로 구성되므로, 일반 분류 Accuracy나 ROC보다 탐지 mAP와 PR을 사용합니다.

In [ ]:
save_and_show(plot_training_report(run_dir), "training_curves.png")

for filename in ("BoxPR_curve.png", "confusion_matrix.png"):
    path = run_dir / filename
    if path.is_file():
        display(NotebookImage(filename=str(path)))
    else:
        print(f"저장된 그림 없음 · {filename}")

## 3. 신뢰도 기준만 바꾼 통제실험

같은 `best.pt`, 검증 프레임, IoU 0.50을 사용하고 신뢰도만 바꿉니다. 낮은 기준에서 한 번 추론해 후보들을 보관합니다. 탐지 매칭은 높은 신뢰도부터 진행하므로 이 단일 클래스 평가에서 기준을 올릴 때는 그 기준 이상의 예측만 셉니다.

표의 FP는 원판이 있는 검증 장면에서 나온 오탐입니다. 원판 없는 장면에서 얼마나 자주 오탐하는지는 별도의 원판 없는 영상으로 측정합니다.

In [ ]:
if any(not minimum_confidence <= value <= 1 for value in thresholds):
    raise ValueError("thresholds는 minimum_confidence 이상, 1 이하여야 합니다.")

model = YOLO(str((run_dir / "weights" / "best.pt").resolve(strict=True)))
review = load_validation_frames(DATASET / "data.yaml", SPLIT)
review.compute_metadata()
evaluate_plates(review, model, confidence=minimum_confidence)

targets, predictions = collect_detections(review)
comparison = compare_thresholds(targets, predictions, thresholds)
display(comparison.round(3))

axis = comparison.set_index("confidence")[["TP", "FP", "FN"]].plot.bar(
    figsize=(6.4, 3.6), rot=0, color=["#2563a6", "#d47729", "#b74949"],
    title="Same model and frames · IoU 0.50",
)
axis.set(xlabel="Confidence threshold", ylabel="Boxes")
for bars in axis.containers:
    axis.bar_label(bars, padding=2)
axis.margins(y=0.12)
save_and_show(axis.figure, "confidence_comparison.png")

## 4. 영상별 신뢰도와 원판 중심 오차

같은 영상에서 신뢰도가 낮아지는 프레임을 보고, 정답에 대응된 박스의 중심이 원본 해상도에서 몇 픽셀 어긋났는지 확인합니다. 미탐 프레임은 중심 오차를 계산할 박스가 없으므로 아래 그래프에 세로선으로 표시합니다. 점은 라벨링한 프레임입니다.

중심 오차는 원판 궤적을 그릴 때 직접 영향을 줍니다. 영상별 촬영 거리와 해상도가 다르면 픽셀 오차를 같은 물리적 거리로 해석하기 어렵습니다.

In [ ]:
video_id = "a03_sn_f_01"
confidence = 0.10
save_and_show(
    plot_frame_errors(targets, predictions, video_id, confidence),
    f"frame_errors_{video_id}.png",
)

## 5. XAI 가림 실험

같은 프레임에서 **원판**, **앞쪽의 원판 랙**, **빈 바닥**을 차례로 가립니다. 세 마스크는 크기와 색이 같고 위치만 다릅니다. 초록 실선은 정답, 노란 점선은 정답과 IoU 0.50 이상으로 겹치는 후보 중 신뢰도가 가장 높은 박스입니다.

어느 영역을 가렸을 때 점수가 크게 달라지는지 봅니다. 회색 가림 자체가 입력을 바꾸므로 이 그림은 해당 프레임에서의 반응을 보여줍니다. 여러 프레임에서 반복해야 일반적인 경향을 판단할 수 있습니다. [Zeiler와 Fergus의 가림 민감도 실험](https://arxiv.org/abs/1311.2901)

In [ ]:
frame_index = 25
stem = f"{video_id}_f{frame_index}"
with Image.open(DATASET / "images" / "val" / f"{stem}.jpg") as image:
    frame = np.array(image.convert("RGB"))

(target,) = read_boxes(DATASET / "labels" / "val" / f"{stem}.txt")
# 좌표는 이미지 왼쪽 위를 (0, 0), 오른쪽 아래를 (1, 1)로 본 마스크 중심입니다.
regions = {
    "Plate": target[:2],
    "Foreground rack": (0.48, 0.77),
    "Floor control": (0.85, 0.66),
}
mask_size = (100, 100)  # 가로, 세로 픽셀

figure, occlusion = plot_occlusion_comparison(
    model, frame, target, regions, mask_size, min_confidence=minimum_confidence
)
display(occlusion.round(3))
save_and_show(figure, f"occlusion_{stem}.png")

## 6. 다음 모델과 비교할 때

비교할 모델마다 실제 학습 사람 기록을 검사하고, 동일한 검증 프레임·신뢰도·IoU에서 오탐·미탐·중심 오차를 확인합니다. 시드나 증강 등 한 조건을 바꿀 때는 변경 항목을 기록합니다.

앙상블을 실험하면 멤버별 결과와 결합 결과를 같은 조건에서 나란히 비교합니다. 결합 규칙이나 모델을 고르는 데 사용한 영상은 최종 성능 평가에 다시 사용하지 않습니다. 현재 검증셋은 개선 방향을 찾는 데 쓰므로, 최종 수치는 튜닝에 쓰지 않은 사람의 영상에서 측정합니다.